In [0]:
# ==============================================================================
# CAMADA GOLD: Analytics e Respostas às Perguntas de Negócio
# ==============================================================================

from pyspark.sql.functions import avg, median, count, round, col, when

# Carregamento das Tabelas do Star Schema
fact = spark.table("workspace.default.fact_listings")
dim_loc = spark.table("workspace.default.dim_location")
dim_prop = spark.table("workspace.default.dim_property")
dim_host = spark.table("workspace.default.dim_host")

# ------------------------------------------------------------------------------
# PERGUNTA 1: Variação de preço médio e mediano por Zona
# ------------------------------------------------------------------------------
print("--- 1. Análise Financeira por Zona Geográfica ---")
p1 = fact.join(dim_loc, "location_id") \
    .groupBy("zone") \
    .agg(
        round(avg("price"), 2).alias("preco_medio"),
        median("price").alias("preco_mediano"),
        count("property_id").alias("total_anuncios")
    ).orderBy(col("preco_mediano").desc())
display(p1)

# ------------------------------------------------------------------------------
# PERGUNTA 2: Impacto de comodidades críticas (Ar-Condicionado e Vista Mar)
# ------------------------------------------------------------------------------
print("--- 2. Impacto das Comodidades no Preço Mediano ---")
p2 = fact.join(dim_prop, "property_id") \
    .groupBy("has_air_conditioning", "has_sea_view") \
    .agg(
        median("price").alias("preco_mediano"),
        round(avg("price"), 2).alias("preco_medio"),
        count("property_id").alias("total_anuncios")
    )
display(p2)

# ------------------------------------------------------------------------------
# PERGUNTA 3: Superhosts possuem notas de avaliação e preços superiores?
# ------------------------------------------------------------------------------
print("--- 3. Comparativo: Anfitriões Comuns vs Superhosts ---")
p3 = fact.join(dim_host, "host_id") \
    .groupBy("is_superhost") \
    .agg(
        round(avg("review_score"), 2).alias("nota_media"),
        median("price").alias("preco_mediano"),
        count("property_id").alias("total_anuncios")
    )
display(p3)

# ------------------------------------------------------------------------------
# PERGUNTA 4: Perfil dos Anfitriões (Profissionais vs Amadores)
# ------------------------------------------------------------------------------
print("--- 4. Concentração de Mercado por Perfil de Anfitrião ---")
p4 = fact.join(dim_host, "host_id") \
    .withColumn("perfil_anfitricao", when(col("host_listings_count") > 1, "Multi-Proprietário (Profissional)").otherwise("Individual (Amador)")) \
    .groupBy("perfil_anfitricao") \
    .agg(
        count("property_id").alias("total_anuncios"),
        round((count("property_id") / fact.count()) * 100, 2).alias("percentual_mercado"),
        median("price").alias("preco_mediano")
    )
display(p4)

# ------------------------------------------------------------------------------
# PERGUNTA 5: Relação entre Mínimo de Noites exigido e Preço
# ------------------------------------------------------------------------------
print("--- 5. Relação entre Exigência de Noites Mínimas e Diária ---")
p5 = fact \
    .withColumn("categoria_estadia", 
        when(col("minimum_nights") <= 2, "Curta (1-2 noites)")
        .when((col("minimum_nights") >= 3) & (col("minimum_nights") <= 7), "Média (3-7 noites)")
        .otherwise("Longa (8+ noites)")
    ) \
    .groupBy("categoria_estadia") \
    .agg(
        median("price").alias("preco_mediano"),
        count("property_id").alias("total_anuncios")
    )
display(p5)